# Machine Failure Prediction (Predictive Maintenance)

**Goal:** Sensor data (temperature, torque, tool wear) se predict karna ki machine fail karegi ya nahi.

**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI Machine Learning Repository) — 10,000 industrial machine readings.

**Kaise use karein:** Upar Menu me `Runtime -> Run all` click karo. Sab cells automatically chal jaayenge. Last me accuracy number milega — wahi resume me daalna hai.

Har cell ke upar comment me bataya gaya hai ki wo cell kya kar raha hai.

## Step 1: Libraries Import Karo
Yeh sab Python ki ready-made libraries hain jo data handle karne aur ML model banane me help karti hain. Google Colab me yeh already installed hoti hain, kuch karna nahi padega.

In [ ]:
import pandas as pd                     # data ko table (DataFrame) ki tarah handle karne ke liye
import numpy as np                      # numerical calculations ke liye
import matplotlib.pyplot as plt         # graphs banane ke liye

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Libraries successfully imported!")

## Step 2: Dataset Load Karo
Yeh dataset directly internet se UCI ki official website se aa jaayega — koi manual download nahi karna.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

print("Dataset shape (rows, columns):", df.shape)
df.head()

## Step 3: Data Samjho
`Machine failure` column hi hamara target hai — 0 matlab machine theek hai, 1 matlab fail hui.

Niche dekho kitne % machines fail hui — yeh number bahut chhota hoga (imbalanced data), isliye sirf accuracy pe bharosa nahi karna, precision/recall bhi dekhna padega (next steps me samjhayenge).

In [ ]:
df.info()
print("\nFailure distribution:")
print(df['Machine failure'].value_counts())
print("\nFailure percentage:", round(df['Machine failure'].mean()*100, 2), "%")

## Step 4: Unnecessary / Leaky Columns Hatao

- `UDI` aur `Product ID` sirf ID hain, prediction me kaam ke nahi — hatao.
- `TWF, HDF, PWF, OSF, RNF` — yeh columns batate hain *kis wajah se* machine fail hui. Yeh humare target (`Machine failure`) ko seedha reveal kar dete hain (data leakage), isliye inhe bhi hatana zaroori hai — warna model 'cheating' kar lega aur real-world me kaam nahi karega.

Yeh ek important ML concept hai — interview me poochha ja sakta hai!

In [ ]:
df_model = df.drop(columns=['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
df_model.head()

## Step 5: Text Column ko Number me Convert Karo
`Type` column me L, M, H (Low/Medium/High quality) likha hai — ML models text nahi samajhte, isliye isko numbers (0,1,2) me convert karte hain.

In [ ]:
le = LabelEncoder()
df_model['Type'] = le.fit_transform(df_model['Type'])   # L=0/1, M=0/1, H=0/1 jaisa kuch ban jaayega

print("Type column ke unique encoded values:", df_model['Type'].unique())
df_model.head()

## Step 6: Data ko Train / Test me Baanto
Model ko train karne ke liye 80% data, aur usko test (check) karne ke liye 20% data alag rakhte hain — taaki pata chale model naye/unseen data pe kaisa perform karta hai.

In [ ]:
X = df_model.drop(columns=['Machine failure'])   # features (input)
y = df_model['Machine failure']                  # target (output jo predict karna hai)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## Step 7: Model Train Karo (Random Forest)
Random Forest kaafi decision trees ka combination hota hai — robust aur accurate, beginners ke liye best choice. `class_weight='balanced'` isliye diya hai kyunki failures bahut kam hain (imbalanced data), yeh model ko rare failures pe extra dhyaan dene me help karta hai.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train, y_train)
print("Model training complete!")

## Step 8: Model Evaluate Karo
Yahi number resume me jaayenge. **Accuracy** ke saath **Precision, Recall, F1-score** bhi dekho — imbalanced data me yeh zyada meaningful hain.

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall   : {rec*100:.2f}%")
print(f"F1-score : {f1*100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nFull Report:")
print(classification_report(y_test, y_pred))

## Step 9 (Bonus): Konsa Feature Sabse Important Hai?
Yeh graph dikhata hai ki model ne prediction karte waqt kis sensor reading ko sabse zyada important maana. Ye screenshot resume/LinkedIn post me bhi use kar sakte ho — recruiters ko impress karta hai.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
importances.plot(kind='bar', color='steelblue')
plt.title('Feature Importance - Machine Failure Prediction')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()

print(importances)

## Next Steps (GitHub par upload karna)

1. Upar `File -> Download -> Download .ipynb` karo (ya isi file ko jo tumhe diya gaya hai, use karo).
2. GitHub.com par jaake naya repository banao, naam do: `machine-failure-prediction`.
3. "Add file -> Upload files" se yeh `.ipynb` file upload kar do.
4. Repository ka link copy karo aur resume me `GitHub Repository` wali line me daal do.
5. Jo Accuracy number Step 8 me aaya, wahi number resume ke project bullet point me update kar do.

**Tip:** README.md bhi add kar do repo me — 3-4 lines me likh do ki project kya karta hai aur konsa dataset use kiya.